In [ ]:
%%bash
set -euo pipefail
cat > /kaggle/working/l3_env.sh <<'EOF'
export W=/tmp/l3
export ENVDIR=$W/envs/bevfusion
export PY=$ENVDIR/bin/python
export C=$W/constraints.txt
export CUDA_VISIBLE_DEVICES=0
export MPLBACKEND=Agg
export OUT=/kaggle/working/outputs
[ -d /usr/local/cuda ] && export CUDA_HOME=/usr/local/cuda
true
EOF
source /kaggle/working/l3_env.sh
mkdir -p $W $OUT
TAR=$(find /kaggle/input -maxdepth 6 -name 'env_bevfusion.tar.*' | head -1)
[ -n "$TAR" ] || { echo "khong tim thay env tar"; find /kaggle/input -maxdepth 4 | head -40; exit 1; }
ls -la "$TAR"
case "$TAR" in *.zst) zstd -dc "$TAR" | tar -C $W -xf - ;; *.gz) tar -C $W -xzf "$TAR" ;; esac
V=$(find /kaggle/input -maxdepth 6 -type d -name v1.0-mini | head -1); ROOT=$(dirname "$V"); echo "nuScenes root: $ROOT"
DST=$W/mmdetection3d/data/nuscenes
for d in samples sweeps maps v1.0-mini; do ln -sfn "$ROOT/$d" "$DST/$d"; done
LC=$(find /kaggle/input -name our_lidar_mini.pth | head -1)
[ -n "$LC" ] || { echo "khong thay our_lidar_mini.pth cua Stage A"; find /kaggle/input -maxdepth 5 | head -60; exit 1; }
ls -la "$LC"; echo "$LC" > $W/our_lidar_ckpt_path.txt
$PY --version

In [ ]:
import os
files = {
  '/tmp/l3/mmdetection3d/projects/BEVFusion/configs/our_bevfusion_mini.py': "# Stage B: BEVFusion Camera+LiDAR. LiDAR branch initialised from OUR Stage-A checkpoint; Swin-T image backbone uses the official\n# ImageNet-pretrained weights (BEVFusion workflow). No official LiDAR / LiDAR+Camera BEVFusion checkpoint is loaded.\nimport os\n\n_base_ = ['./bevfusion_lidar-cam_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d.py']\n\nEPOCHS = int(os.environ.get('OUR_EPOCHS', 6))\nLR = float(os.environ.get('OUR_LR', 3e-5))\nPEAK = LR * 3.5          # ~1e-4 for effective batch 4 (official: 2e-4 for batch 32)\nACCUM = int(os.environ.get('OUR_ACCUM', 4))\nVAL_INT = int(os.environ.get('OUR_VAL_INT', 1))\nNUM_WORKERS = int(os.environ.get('OUR_WORKERS', 4))\nWITH_CP = bool(int(os.environ.get('OUR_WITH_CP', '0')))\nWARM_EPOCHS = EPOCHS * 0.4\n\nload_from = os.environ['OUR_LIDAR_CKPT']\nresume = False\nrandomness = dict(seed=42, deterministic=False)\n\nmodel = dict(img_backbone=dict(with_cp=WITH_CP))\n\ntrain_dataloader = dict(batch_size=1, num_workers=NUM_WORKERS, persistent_workers=True)\nval_dataloader = dict(batch_size=1, num_workers=2, dataset=dict(metainfo=dict(version='v1.0-mini')))\ntest_dataloader = dict(batch_size=1, num_workers=2, dataset=dict(metainfo=dict(version='v1.0-mini')))\n\noptim_wrapper = dict(optimizer=dict(lr=PEAK), accumulative_counts=ACCUM)\n\nparam_scheduler = [\n    dict(type='LinearLR', start_factor=0.33333333, by_epoch=False, begin=0, end=200),\n    dict(type='CosineAnnealingLR', begin=0, T_max=EPOCHS, end=EPOCHS, by_epoch=True, eta_min_ratio=1e-4,\n         convert_to_iter_based=True),\n    dict(type='CosineAnnealingMomentum', eta_min=0.85 / 0.95, begin=0, end=WARM_EPOCHS, by_epoch=True,\n         convert_to_iter_based=True),\n    dict(type='CosineAnnealingMomentum', eta_min=1, begin=WARM_EPOCHS, end=EPOCHS, by_epoch=True,\n         convert_to_iter_based=True)\n]\n\ntrain_cfg = dict(by_epoch=True, max_epochs=EPOCHS, val_interval=VAL_INT)\n\ndefault_hooks = dict(\n    logger=dict(type='LoggerHook', interval=20),\n    checkpoint=dict(type='CheckpointHook', interval=1, max_keep_ckpts=-1, save_optimizer=True,\n                    save_best='NuScenes metric/pred_instances_3d_NuScenes/NDS', rule='greater'))\n",
  '/tmp/l3/train_driver.py': '"""Smoke-train -> verify -> full-train -> evaluate -> export driver for BEVFusion on nuScenes-mini (runs inside the frozen env)."""\nimport glob, json, os, re, shutil, subprocess, sys, time\nimport numpy as np\n\nW = os.environ.get(\'W\', \'/tmp/l3\')\nPY = os.environ[\'PY\']\nOUT = os.environ.get(\'OUT\', \'/kaggle/working/outputs\')\nMMD = f\'{W}/mmdetection3d\'\nCFG = os.environ.get(\'OUR_CFG\', \'projects/BEVFusion/configs/our_lidar_mini.py\')\nNAME = os.environ.get(\'OUR_NAME\', \'our_lidar_mini\')\nBUDGET_S = float(os.environ.get(\'OUR_BUDGET_S\', 11520))\nSTAGE_DIR = os.path.join(OUT, NAME)\nKEY_NDS = \'NuScenes metric/pred_instances_3d_NuScenes/NDS\'\nKEY_MAP = \'NuScenes metric/pred_instances_3d_NuScenes/mAP\'\nos.makedirs(f\'{W}/logs\', exist_ok=True)\nos.makedirs(STAGE_DIR, exist_ok=True)\n\n\ndef log(*a):\n    print(time.strftime(\'%H:%M:%S\'), *a, flush=True)\n\n\ndef tail_progress(path):\n    try:\n        with open(path, \'rb\') as f:\n            f.seek(0, 2); n = f.tell(); f.seek(max(0, n - 6000))\n            txt = f.read().decode(\'utf-8\', \'replace\')\n        lines = [l for l in txt.splitlines() if \'Epoch(\' in l or \'Traceback\' in l or \'Error\' in l]\n        if lines:\n            print(\'   ..\', lines[-1][-260:], flush=True)\n    except Exception:\n        pass\n\n\ndef run(cmd, extra_env, logfile, every=300):\n    env = dict(os.environ)\n    env.update({k: str(v) for k, v in extra_env.items()})\n    env[\'PYTHONPATH\'] = MMD + \':\' + env.get(\'PYTHONPATH\', \'\')\n    with open(logfile, \'w\') as f:\n        p = subprocess.Popen(cmd, cwd=MMD, env=env, stdout=f, stderr=subprocess.STDOUT)\n        last = 0\n        while p.poll() is None:\n            time.sleep(10)\n            if time.time() - last > every:\n                last = time.time(); tail_progress(logfile)\n    tail_progress(logfile)\n    return p.returncode\n\n\ndef train(tag, epochs, lr, accum, val_int, amp, extra=None):\n    wd = f\'{W}/work_dirs/{tag}\'\n    shutil.rmtree(wd, ignore_errors=True)\n    cmd = [PY, \'-W\', \'ignore\', \'tools/train.py\', CFG, \'--work-dir\', wd] + ([\'--amp\'] if amp else [])\n    env = dict(OUR_EPOCHS=epochs, OUR_LR=lr, OUR_ACCUM=accum, OUR_VAL_INT=val_int)\n    env.update(extra or {})\n    lf = f\'{W}/logs/train_{tag}.log\'\n    t0 = time.time()\n    rc = run(cmd, env, lf)\n    return dict(rc=rc, wd=wd, log=lf, secs=time.time() - t0, tag=tag)\n\n\ndef load_scalars(wd):\n    fs = sorted(glob.glob(f\'{wd}/*/vis_data/scalars.json\'))\n    if not fs:\n        return None, []\n    recs = []\n    for l in open(fs[-1]):\n        l = l.strip()\n        if l:\n            try:\n                recs.append(json.loads(l))\n            except Exception:\n                pass\n    return fs[-1], recs\n\n\ndef summarize(wd, log_path, epochs):\n    sp, recs = load_scalars(wd)\n    tr = [r for r in recs if \'loss\' in r]\n    val = [r for r in recs if KEY_NDS in r]\n    txt = open(log_path, errors=\'replace\').read()\n    m = re.findall(r\'Epoch\\(train\\)\\s+\\[1\\]\\[\\s*\\d+/(\\d+)\\]\', txt)\n    iters_per_epoch = int(m[-1]) if m else None\n    rep = dict(n_train_records=len(tr), n_val_records=len(val), iters_per_epoch=iters_per_epoch,\n               oom=(\'out of memory\' in txt.lower()), scalars=sp)\n    if not tr:\n        rep[\'ok\'] = False; rep[\'why\'] = \'no train records\'; return rep, tr, val\n    loss = np.array([r[\'loss\'] for r in tr], float)\n    gn = np.array([r.get(\'grad_norm\', 0.0) for r in tr], float)\n    q = max(1, len(loss) // 4)\n    rep.update(loss_finite=bool(np.isfinite(loss).all()), gradnorm_nonfinite_frac=float(np.mean(~np.isfinite(gn))),\n               loss_first_quarter=float(np.nanmean(loss[:q])), loss_last_quarter=float(np.nanmean(loss[-q:])),\n               sec_per_iter=float(np.median([r[\'time\'] for r in tr if \'time\' in r])),\n               max_mem_mb=float(max([r.get(\'memory\', 0) for r in tr])), last_lr=float(tr[-1].get(\'lr\', 0)))\n    ck = os.path.join(wd, f\'epoch_{epochs}.pth\')\n    rep[\'ckpt_exists\'] = os.path.exists(ck)\n    exp_load = os.environ.get(\'OUR_EXPECT_LOAD\', \'\')\n    low = txt.lower()\n    if exp_load:\n        rep[\'load_ok\'] = (exp_load in txt) and (\'size mismatch\' not in low)\n    else:\n        rep[\'load_ok\'] = (\'load checkpoint from\' not in low)\n    rep[\'ok\'] = bool(rep[\'loss_finite\'] and rep[\'gradnorm_nonfinite_frac\'] < 0.2 and rep[\'ckpt_exists\'] and rep[\'load_ok\']\n                     and rep[\'loss_last_quarter\'] < rep[\'loss_first_quarter\'])\n    return rep, tr, val\n\n\ndef evaluate(ckpt, tag):\n    exp = f\'{W}/exports/{tag}\'\n    shutil.rmtree(exp, ignore_errors=True)\n    lf = f\'{W}/logs/eval_{tag}.log\'\n    cmd = [PY, \'-W\', \'ignore\', \'tools/test.py\', CFG, ckpt, \'--work-dir\', f\'{W}/work_dirs/eval_{tag}\',\n           \'--cfg-options\', f\'test_evaluator.jsonfile_prefix={exp}\']\n    rc = run(cmd, {}, lf, every=120)\n    txt = open(lf, errors=\'replace\').read()\n    mism = (\'size mismatch\' in txt) or (\'missing keys in source state_dict\' in txt) or (\'unexpected key in source state_dict\' in txt)\n    ms = glob.glob(f\'{exp}/**/metrics_summary.json\', recursive=True)\n    res = glob.glob(f\'{exp}/**/results_nusc.json\', recursive=True)\n    out = dict(tag=tag, ckpt=os.path.basename(ckpt), rc=rc, load_mismatch=mism)\n    if ms:\n        m = json.load(open(ms[0]))\n        out.update(mAP=m[\'mean_ap\'], NDS=m[\'nd_score\'], mATE=m[\'tp_errors\'][\'trans_err\'], mASE=m[\'tp_errors\'][\'scale_err\'],\n                   mAOE=m[\'tp_errors\'][\'orient_err\'], mAVE=m[\'tp_errors\'][\'vel_err\'], mAAE=m[\'tp_errors\'][\'attr_err\'],\n                   per_class_AP_2m={k: v for k, v in m[\'mean_dist_aps\'].items()})\n        out[\'results_json\'] = res[0] if res else None\n    out[\'ok\'] = bool(ms) and rc == 0 and not mism\n    return out\n\n\ndef strip(ckpt, dst, note):\n    import torch\n    c = torch.load(ckpt, map_location=\'cpu\')\n    keep = dict(state_dict=c[\'state_dict\'], meta=c.get(\'meta\', {}))\n    keep[\'meta\'][\'our_note\'] = note\n    torch.save(keep, dst)\n    return os.path.getsize(dst) / 1e6\n\n\ndef curves(tr, val, outdir, title):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    os.makedirs(outdir, exist_ok=True)\n    it = np.arange(len(tr)) * 1.0\n    loss = np.array([r[\'loss\'] for r in tr])\n    k = 10\n    sm = np.convolve(loss, np.ones(k) / k, mode=\'valid\') if len(loss) > k else loss\n    fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))\n    ax[0].plot(it, loss, alpha=.3, label=\'loss (logged every 20 it)\'); ax[0].plot(np.arange(len(sm)) + k / 2, sm, label=\'moving avg\')\n    ax[0].set_title(title + \' | total loss\'); ax[0].set_xlabel(\'log step\'); ax[0].legend(); ax[0].set_yscale(\'log\')\n    for key in (\'loss_heatmap\', \'loss_cls\', \'loss_bbox\'):\n        v = [r.get(key) for r in tr if key in r]\n        if v:\n            ax[1].plot(v, label=key, alpha=.8)\n    lay = [k2 for k2 in tr[0].keys() if k2.endswith(\'_loss_cls\') or k2.endswith(\'loss_cls\')]\n    ax[1].set_title(\'loss components\'); ax[1].legend(); ax[1].set_yscale(\'log\')\n    fig.tight_layout(); fig.savefig(f\'{outdir}/loss.png\', dpi=100); plt.close(fig)\n    fig, ax = plt.subplots(figsize=(7, 4)); ax.plot([r.get(\'lr\', 0) for r in tr]); ax.set_title(\'learning rate\'); ax.set_xlabel(\'log step\')\n    fig.tight_layout(); fig.savefig(f\'{outdir}/lr.png\', dpi=100); plt.close(fig)\n    if val:\n        fig, ax = plt.subplots(figsize=(7, 4))\n        st = [r.get(\'step\', i) for i, r in enumerate(val)]\n        ax.plot(st, [r[KEY_NDS] for r in val], \'o-\', label=\'NDS\'); ax.plot(st, [r[KEY_MAP] for r in val], \'s-\', label=\'mAP\')\n        ax.set_title(\'mini VAL metrics vs epoch\'); ax.set_xlabel(\'epoch\'); ax.legend(); ax.grid(alpha=.3)\n        fig.tight_layout(); fig.savefig(f\'{outdir}/val_metrics.png\', dpi=100); plt.close(fig)\n\n\nDRYRUN_SCRIPT = r"""\nimport sys, os, importlib, numpy as np\nsys.path.insert(0, os.getcwd())\nfrom mmengine import Config\nfrom mmdet3d.utils import register_all_modules\nregister_all_modules(init_default_scope=True)\nfrom mmdet3d.registry import DATASETS\ncfg = Config.fromfile(os.environ[\'OUR_CFG\'])\nfor m in cfg.get(\'custom_imports\', {}).get(\'imports\', []):\n    importlib.import_module(m)\nds = DATASETS.build(cfg.train_dataloader.dataset)\nn = len(ds)\nprint(\'train dataset len (CBGS-resampled):\', n)\nfor i in np.linspace(0, n - 1, 24).astype(int):\n    it = ds[int(i)]\n    p = it[\'inputs\'][\'points\']\n    g = it[\'data_samples\'].gt_instances_3d\n    im = it[\'inputs\'].get(\'img\')\n    print(int(i), \'points\', tuple(p.shape), \'gt boxes\', len(g.labels_3d), \'img\', None if im is None else tuple(im.shape), flush=True)\nprint(\'DRYRUN PASS\')\n"""\n\n\ndef dryrun():\n    env = dict(os.environ)\n    env[\'PYTHONPATH\'] = MMD + \':\' + env.get(\'PYTHONPATH\', \'\')\n    env[\'OUR_CFG\'] = CFG\n    r = subprocess.run([PY, \'-W\', \'ignore\', \'-c\', DRYRUN_SCRIPT], cwd=MMD, env=env, capture_output=True, text=True)\n    out = (r.stdout or \'\') + (r.stderr or \'\')\n    print(out[-3500:], flush=True)\n    if \'DRYRUN PASS\' not in out:\n        log(\'DRYRUN FAIL (train data pipeline broken; hyper-parameter retries would be pointless)\')\n        sys.exit(4)\n\n\ndef main():\n    smoke_only = os.environ.get(\'OUR_SMOKE_ONLY\') == \'1\'\n    attempts = [dict(amp=True, lr=3e-5, accum=4, extra={}),\n                dict(amp=True, lr=3e-5, accum=4, extra={\'PYTORCH_CUDA_ALLOC_CONF\': \'expandable_segments:True\', \'OUR_WORKERS\': 2}),\n                dict(amp=True, lr=3e-5, accum=4, extra={\'PYTORCH_CUDA_ALLOC_CONF\': \'expandable_segments:True\', \'OUR_WITH_CP\': 1, \'OUR_WORKERS\': 2}),\n                dict(amp=True, lr=1.5e-5, accum=4, extra={\'PYTORCH_CUDA_ALLOC_CONF\': \'expandable_segments:True\'}),\n                dict(amp=False, lr=1.5e-5, accum=4, extra={\'PYTORCH_CUDA_ALLOC_CONF\': \'expandable_segments:True\'})]\n    chosen, smoke_reports = None, []\n    dryrun()\n    for i, a in enumerate(attempts):\n        log(f\'=== SMOKE attempt {i}: {a}\')\n        r = train(f\'smoke{i}\', 2, a[\'lr\'], a[\'accum\'], 1, a[\'amp\'], a[\'extra\'])\n        rep, tr, val = summarize(r[\'wd\'], r[\'log\'], 2)\n        rep.update(attempt=i, params={k: v for k, v in a.items()}, rc=r[\'rc\'], secs=r[\'secs\'])\n        if rep.get(\'ckpt_exists\'):\n            ev = evaluate(os.path.join(r[\'wd\'], \'epoch_2.pth\'), f\'smoke{i}\')\n            rep[\'reload_eval\'] = {k: v for k, v in ev.items() if k != \'per_class_AP_2m\'}\n            rep[\'ok\'] = bool(rep[\'ok\'] and ev[\'ok\'])\n        log(\'smoke report:\', json.dumps({k: v for k, v in rep.items() if k not in (\'params\',)}, default=str)[:1500])\n        smoke_reports.append(rep)\n        if rep[\'ok\']:\n            chosen = (a, rep); break\n        log(\'smoke attempt failed; tail of log:\'); print(open(r[\'log\'], errors=\'replace\').read()[-2500:])\n        if rep.get(\'why\') == \'no train records\' and not rep[\'oom\']:\n            log(\'non-OOM crash: retrying with other hyper-parameters is pointless -> stop\'); break\n    os.makedirs(f\'{STAGE_DIR}/training_logs\', exist_ok=True)\n    json.dump(smoke_reports, open(f\'{STAGE_DIR}/training_logs/smoke_report.json\', \'w\'), indent=2, default=str)\n    if not chosen:\n        log(\'SMOKE FAIL\'); sys.exit(2)\n    a, rep = chosen\n    log(\'SMOKE PASS with\', a)\n    print(\'SMOKE PASS\')\n    if smoke_only:\n        return\n    ipe, spi = rep[\'iters_per_epoch\'], rep[\'sec_per_iter\']\n    epoch_s = (ipe * spi) if (ipe and spi) else 600.0\n    val_int = 2\n    E = int(np.clip(BUDGET_S / (epoch_s * 1.08), float(os.environ.get(\'OUR_EMIN\', 12)), float(os.environ.get(\'OUR_EMAX\', 60))))\n    log(f\'iters/epoch={ipe} sec/iter={spi:.3f} epoch~{epoch_s / 60:.1f} min -> full schedule EPOCHS={E}, val_interval={val_int}\')\n    r = train(\'full\', E, a[\'lr\'], a[\'accum\'], val_int, a[\'amp\'], a[\'extra\'])\n    rep_full, tr, val = summarize(r[\'wd\'], r[\'log\'], E)\n    rep_full.update(rc=r[\'rc\'], secs=r[\'secs\'], epochs=E, params=a)\n    json.dump(rep_full, open(f\'{STAGE_DIR}/training_logs/full_train_report.json\', \'w\'), indent=2, default=str)\n    if not rep_full.get(\'ckpt_exists\'):\n        print(open(r[\'log\'], errors=\'replace\').read()[-3000:]); log(\'FULL TRAIN FAIL\'); sys.exit(3)\n    finish(r[\'wd\'], r[\'log\'], E, tr, val, rep_full)\n\n\ndef finish(wd, logf, E, tr, val, rep_full):\n    os.makedirs(f\'{STAGE_DIR}/training_logs\', exist_ok=True)\n    shutil.copy(logf, f\'{STAGE_DIR}/training_logs/train_full.log\')\n    sp, _ = load_scalars(wd)\n    if sp:\n        shutil.copy(sp, f\'{STAGE_DIR}/training_logs/scalars.json\')\n    curves(tr, val, f\'{STAGE_DIR}/training_curves\', NAME)\n    last = os.path.join(wd, f\'epoch_{E}.pth\')\n    note = f\'{NAME}: trained from scratch on nuScenes-mini TRAIN (no official checkpoint), epochs={E}\'\n    mb = strip(last, f\'{STAGE_DIR}/{NAME}.pth\', note)\n    log(f\'{NAME}.pth written ({mb:.0f} MB)\')\n    os.makedirs(f\'{STAGE_DIR}/epoch_ckpts\', exist_ok=True)\n    for e in range(1, E + 1):\n        p = os.path.join(wd, f\'epoch_{e}.pth\')\n        if os.path.exists(p) and (e == 1 or e % 5 == 0):\n            strip(p, f\'{STAGE_DIR}/epoch_ckpts/epoch_{e}.pth\', note)\n    ev_last = evaluate(f\'{STAGE_DIR}/{NAME}.pth\', \'final_last\')\n    best = sorted(glob.glob(f\'{wd}/best_*.pth\'))\n    ev_best = None\n    if best:\n        strip(best[-1], f\'{STAGE_DIR}/{NAME}_best_by_val_NDS.pth\', note + \' (selected by val NDS: optimistic)\')\n        ev_best = evaluate(f\'{STAGE_DIR}/{NAME}_best_by_val_NDS.pth\', \'final_best\')\n    for ev in (ev_last, ev_best):\n        if ev and ev.get(\'results_json\'):\n            d = os.path.join(STAGE_DIR, \'predictions\', ev[\'tag\']); os.makedirs(d, exist_ok=True)\n            shutil.copy(ev[\'results_json\'], d)\n    per_epoch = [dict(epoch=r.get(\'step\'), mAP=r.get(KEY_MAP), NDS=r.get(KEY_NDS)) for r in val]\n    metrics = dict(name=NAME, note=\'mini VAL (81 samples, 2 scenes); NOT a full nuScenes benchmark\', epochs=E,\n                   train_secs=rep_full.get(\'secs\'), iters_per_epoch=rep_full.get(\'iters_per_epoch\'), params=rep_full.get(\'params\'),\n                   final_last_epoch=ev_last, best_by_val=ev_best, per_epoch_val=per_epoch,\n                   final_train_loss=rep_full.get(\'loss_last_quarter\'), initial_train_loss=rep_full.get(\'loss_first_quarter\'))\n    json.dump(metrics, open(f\'{STAGE_DIR}/metrics.json\', \'w\'), indent=2, default=str)\n    log(\'FINAL last-epoch:\', {k: ev_last.get(k) for k in (\'mAP\', \'NDS\', \'ok\', \'load_mismatch\')})\n    print(\'STAGE TRAIN PASS\' if ev_last[\'ok\'] else \'STAGE TRAIN FAIL\')\n\n\nif __name__ == \'__main__\':\n    main()\n',
  '/tmp/l3/viz_l3.py': '"""Visualize REAL nuScenes-mini val samples: 6 cameras + 3D pred/GT boxes + LiDAR BEV, for two official models.\nNo nuscenes-devkit needed: reads the json tables directly. Usage: see argparse below."""\nimport argparse, csv, json, os, sys, zipfile\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.gridspec import GridSpec\nfrom PIL import Image\n\nap = argparse.ArgumentParser()\nap.add_argument(\'--root\', required=True, help=\'nuScenes-mini root (has samples/ sweeps/ v1.0-mini/)\')\nap.add_argument(\'--tables\', default=None, help=\'dir with json tables (default: <root>/v1.0-mini)\')\nap.add_argument(\'--res_lidar\', required=True)\nap.add_argument(\'--res_cam\', required=True)\nap.add_argument(\'--out\', required=True)\nap.add_argument(\'--thr\', type=float, default=0.3)\nap.add_argument(\'--limit\', type=int, default=0)\nap.add_argument(\'--mock\', action=\'store_true\', help=\'local code-path test: fake images/points if files are missing\')\nap.add_argument(\'--dpi\', type=int, default=75)\nap.add_argument(\'--every\', type=int, default=1)\nap.add_argument(\'--label_l\', default=\'L3C official LiDAR-only\')\nap.add_argument(\'--label_c\', default=\'L3D official LiDAR+Camera\')\nap.add_argument(\'--note\', default=\'official pretrained BEVFusion on nuScenes-mini VAL (contaminated, not a benchmark)\')\nargs = ap.parse_args()\nTAB = args.tables or os.path.join(args.root, \'v1.0-mini\')\nos.makedirs(args.out, exist_ok=True)\n\n\ndef load_table(name):\n    p = os.path.join(TAB, name + \'.json\')\n    if os.path.exists(p):\n        return json.load(open(p, encoding=\'utf-8\'))\n    with zipfile.ZipFile(p + \'.zip\') as z:\n        return json.load(z.open(z.namelist()[0]))\n\n\ndef qR(q):\n    w, x, y, z = q\n    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],\n                     [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],\n                     [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])\n\n\nsample = {s[\'token\']: s for s in load_table(\'sample\')}\nsd_all = load_table(\'sample_data\')\nego = {e[\'token\']: e for e in load_table(\'ego_pose\')}\ncs = {c[\'token\']: c for c in load_table(\'calibrated_sensor\')}\nscene = {s[\'token\']: s for s in load_table(\'scene\')}\ninst = {i[\'token\']: i for i in load_table(\'instance\')}\ncat = {c[\'token\']: c[\'name\'] for c in load_table(\'category\')}\nkf = {}\nfor d in sd_all:\n    if d[\'is_key_frame\']:\n        kf[(d[\'sample_token\'], d[\'filename\'].split(\'/\')[1])] = d\nann = {}\nfor a in load_table(\'sample_annotation\'):\n    ann.setdefault(a[\'sample_token\'], []).append(a)\n\nGEN2DET = {\'vehicle.car\': \'car\', \'vehicle.truck\': \'truck\', \'vehicle.bus.bendy\': \'bus\', \'vehicle.bus.rigid\': \'bus\',\n           \'vehicle.trailer\': \'trailer\', \'vehicle.construction\': \'construction_vehicle\',\n           \'vehicle.motorcycle\': \'motorcycle\', \'vehicle.bicycle\': \'bicycle\',\n           \'human.pedestrian.adult\': \'pedestrian\', \'human.pedestrian.child\': \'pedestrian\',\n           \'human.pedestrian.construction_worker\': \'pedestrian\', \'human.pedestrian.police_officer\': \'pedestrian\',\n           \'movable_object.trafficcone\': \'traffic_cone\', \'movable_object.barrier\': \'barrier\'}\n\nR_l = json.load(open(args.res_lidar))[\'results\']\nR_c = json.load(open(args.res_cam))[\'results\']\ntokens = sorted(R_l.keys(), key=lambda t: (sample[t][\'scene_token\'], sample[t][\'timestamp\']))\nassert set(R_l) == set(R_c), \'two result files cover different samples\'\ntokens = tokens[::args.every]\nif args.limit:\n    tokens = tokens[:args.limit]\n\nXC = np.array([1, 1, 1, 1, -1, -1, -1, -1.])\nYC = np.array([1, -1, -1, 1, 1, -1, -1, 1.])\nZC = np.array([1, 1, -1, -1, 1, 1, -1, -1.])\nEDGES = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7)]\n\n\ndef corners(center, size, quat):\n    w, l, h = size\n    c = np.stack([XC * l / 2, YC * w / 2, ZC * h / 2])\n    return (qR(quat) @ c).T + np.array(center)\n\n\ndef gt_boxes(t):\n    out = []\n    for a in ann.get(t, []):\n        n = cat[inst[a[\'instance_token\']][\'category_token\']]\n        if n in GEN2DET:\n            out.append(dict(translation=a[\'translation\'], size=a[\'size\'], rotation=a[\'rotation\'],\n                            detection_name=GEN2DET[n], detection_score=1.0))\n    return out\n\n\ndef to_ego(P, e):\n    return (qR(e[\'rotation\']).T @ (P - np.array(e[\'translation\'])).T).T\n\n\ndef to_cam(Pe, c):\n    return (qR(c[\'rotation\']).T @ (Pe - np.array(c[\'translation\'])).T).T\n\n\ndef draw_cam(ax, img, boxes, sd, color, lw, label_scores):\n    c = cs[sd[\'calibrated_sensor_token\']]\n    e = ego[sd[\'ego_pose_token\']]\n    K = np.array(c[\'camera_intrinsic\'])\n    H, W = img.shape[:2]\n    s = W / 1600.0\n    if not ax.images:\n        ax.imshow(img)\n    n_vis = 0\n    for b in boxes:\n        P = corners(b[\'translation\'], b[\'size\'], b[\'rotation\'])\n        Pc = to_cam(to_ego(P, e), c)\n        if np.all(Pc[:, 2] < 0.1):\n            continue\n        uv = (K @ Pc.T).T\n        z = np.maximum(uv[:, 2:3], 1e-3)\n        uv = uv[:, :2] / z * s\n        drawn = False\n        for i, j in EDGES:\n            if Pc[i, 2] > 0.1 and Pc[j, 2] > 0.1:\n                ax.plot([uv[i, 0], uv[j, 0]], [uv[i, 1], uv[j, 1]], color=color, lw=lw)\n                drawn = True\n        if drawn:\n            n_vis += 1\n            if label_scores:\n                ok = np.where(Pc[:, 2] > 0.1)[0]\n                p = uv[ok].mean(0)\n                if 0 <= p[0] < W and 0 <= p[1] < H:\n                    ax.text(p[0], p[1], \'%s %.2f\' % (b[\'detection_name\'][:4], b[\'detection_score\']), color=\'yellow\',\n                            fontsize=5, ha=\'center\', va=\'center\')\n    ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.axis(\'off\')\n    return n_vis\n\n\ndef draw_bev(ax, pts, boxes_pred, boxes_gt, e, title):\n    ax.set_facecolor(\'black\')\n    ax.scatter(-pts[:, 1], pts[:, 0], s=0.25, c=np.clip(pts[:, 2], -3, 3), cmap=\'Greys_r\', vmin=-3, vmax=3, linewidths=0)\n    for boxes, col, lw in ((boxes_gt, \'#39ff14\', 1.0), (boxes_pred, \'#ff3030\', 1.1)):\n        for b in boxes:\n            P = to_ego(corners(b[\'translation\'], b[\'size\'], b[\'rotation\']), e)\n            poly = P[[0, 1, 5, 4, 0]]\n            ax.plot(-poly[:, 1], poly[:, 0], color=col, lw=lw)\n            f = (P[0] + P[1]) / 2\n            ax.plot([-(P[:4, 1].mean()), -f[1]], [P[:4, 0].mean(), f[0]], color=col, lw=lw)\n    ax.plot(0, 0, marker=\'^\', color=\'cyan\', ms=6)\n    ax.set_xlim(-54, 54); ax.set_ylim(-54, 54); ax.set_aspect(\'equal\')\n    ax.tick_params(labelsize=6, colors=\'gray\')\n    ax.set_title(title, fontsize=8)\n\n\nCAMS = [[\'CAM_FRONT_LEFT\', \'CAM_FRONT\', \'CAM_FRONT_RIGHT\'], [\'CAM_BACK_LEFT\', \'CAM_BACK\', \'CAM_BACK_RIGHT\']]\nstats = []\nfor idx, t in enumerate(tokens, 1):\n    s = sample[t]\n    sc = scene[s[\'scene_token\']]\n    lid = kf[(t, \'LIDAR_TOP\')]\n    e_l = ego[lid[\'ego_pose_token\']]\n    c_l = cs[lid[\'calibrated_sensor_token\']]\n    fp = os.path.join(args.root, lid[\'filename\'])\n    if os.path.exists(fp):\n        pts = np.fromfile(fp, dtype=np.float32).reshape(-1, 5)[:, :3]\n    elif args.mock:\n        pts = np.random.randn(20000, 3) * [20, 20, 1]\n    else:\n        raise FileNotFoundError(fp)\n    pts_e = (qR(c_l[\'rotation\']) @ pts.T).T + np.array(c_l[\'translation\'])\n    pl = [b for b in R_l[t] if b[\'detection_score\'] >= args.thr]\n    pc = [b for b in R_c[t] if b[\'detection_score\'] >= args.thr]\n    gt = gt_boxes(t)\n\n    fig = plt.figure(figsize=(24, 9.6))\n    gs = GridSpec(2, 5, figure=fig, width_ratios=[1, 1, 1, 1.05, 1.05], wspace=0.04, hspace=0.06)\n    n_img, n_pred_vis, cam_ok = 0, 0, 0\n    for r in range(2):\n        for k in range(3):\n            ch = CAMS[r][k]\n            d = kf[(t, ch)]\n            ip = os.path.join(args.root, d[\'filename\'])\n            ax = fig.add_subplot(gs[r, k])\n            if os.path.exists(ip):\n                im = Image.open(ip).convert(\'RGB\')\n                im = im.resize((640, 360))\n                img = np.asarray(im); cam_ok += 1\n            elif args.mock:\n                img = np.full((360, 640, 3), 90, np.uint8)\n            else:\n                raise FileNotFoundError(ip)\n            draw_cam(ax, img, gt, d, \'#39ff14\', 0.6, False)\n            n_pred_vis += draw_cam(ax, img, pc, d, \'#ff3030\', 1.0, True)\n            ax.set_title(ch, fontsize=8, color=\'black\', pad=2)\n    ax_l = fig.add_subplot(gs[:, 3])\n    draw_bev(ax_l, pts_e, pl, gt, e_l, args.label_l + \' | %d pred (score>=%.2f) vs %d GT\' % (len(pl), args.thr, len(gt)))\n    ax_c = fig.add_subplot(gs[:, 4])\n    draw_bev(ax_c, pts_e, pc, gt, e_l, args.label_c + \' | %d pred (score>=%.2f) vs %d GT\' % (len(pc), args.thr, len(gt)))\n    head = (\'sample %d/%d | %s (%s) | token %s | t=%.3fs | LiDAR pts %d\\n\'\n            % (idx, len(tokens), sc[\'name\'], sc[\'description\'][:38], t[:8], s[\'timestamp\'] / 1e6, len(pts)))\n    fig.suptitle(head + \'cameras: GT green, right-panel model pred red (yellow = class score) | BEV: GT green, pred red | \' + args.note,\n                 fontsize=9)\n    out = os.path.join(args.out, \'sample_%03d_%s.png\' % (idx, sc[\'name\']))\n    fig.savefig(out, dpi=args.dpi, bbox_inches=\'tight\')\n    plt.close(fig)\n    stats.append(dict(idx=idx, token=t, scene=sc[\'name\'], cams_found=cam_ok, lidar_pts=int(len(pts)), gt=len(gt),\n                      pred_lidar=len(pl), pred_cam=len(pc), file=os.path.basename(out)))\n    if idx % 10 == 0 or idx == len(tokens):\n        print(\'rendered\', idx, \'/\', len(tokens), flush=True)\n\nwith open(os.path.join(args.out, \'viz_index.csv\'), \'w\', newline=\'\') as f:\n    w = csv.DictWriter(f, fieldnames=list(stats[0].keys()))\n    w.writeheader(); w.writerows(stats)\nsummary = dict(n_samples=len(stats), thr=args.thr, cams_missing=sum(6 - x[\'cams_found\'] for x in stats),\n               mean_gt=float(np.mean([x[\'gt\'] for x in stats])), mean_pred_lidar=float(np.mean([x[\'pred_lidar\'] for x in stats])),\n               mean_pred_cam=float(np.mean([x[\'pred_cam\'] for x in stats])), mock=args.mock)\njson.dump(summary, open(os.path.join(args.out, \'viz_summary.json\'), \'w\'), indent=2)\npick = [stats[i][\'file\'] for i in np.linspace(0, len(stats) - 1, min(6, len(stats))).astype(int)]\nims = [Image.open(os.path.join(args.out, f)) for f in pick]\nWm = 1200\nims = [im.resize((Wm, int(im.height * Wm / im.width))) for im in ims]\nsheet = Image.new(\'RGB\', (2 * Wm, sum(im.height for im in ims[0::2]) + 0), \'white\')\ny = [0, 0]\nfor i, im in enumerate(ims):\n    sheet.paste(im, ((i % 2) * Wm, y[i % 2])); y[i % 2] += im.height\nsheet.save(os.path.join(args.out, \'overview_6_samples.png\'))\nprint(json.dumps(summary, indent=2))\nok = summary[\'n_samples\'] == (args.limit or len(tokens)) and (args.mock or summary[\'cams_missing\'] == 0)\nprint(\'VIZ PASS\' if ok else \'VIZ FAIL\')\n',
}
for p, s in files.items():
    open(p, 'w', encoding='utf-8').write(s); print('written', p, len(s))

In [ ]:
import re, glob, subprocess, os
pat = re.compile(r"\bnp\.(long|int|float|bool|object)\b")
rep = {"long": "np.int64", "int": "int", "float": "float", "bool": "bool", "object": "object"}
changed = []
for f in (glob.glob("/tmp/l3/mmdetection3d/mmdet3d/**/*.py", recursive=True) + glob.glob("/tmp/l3/mmdetection3d/projects/BEVFusion/**/*.py", recursive=True)
          + glob.glob("/tmp/l3/mmdetection3d/tools/**/*.py", recursive=True)):
    s = open(f, encoding="utf-8").read()
    n = pat.sub(lambda m: rep[m.group(1)], s)
    if n != s:
        open(f, "w", encoding="utf-8").write(n); changed.append(f)
print("numpy-alias patch (np.long->np.int64; np.int/float/bool/object -> builtins) applied to:", changed)
os.makedirs("/kaggle/working/outputs/config", exist_ok=True)
d = subprocess.run(["git", "-C", "/tmp/l3/mmdetection3d", "diff"], capture_output=True, text=True).stdout
open("/kaggle/working/outputs/config/numpy_alias_patch.diff", "w").write(d)
print(d[:3000])

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/l3_env.sh
export OUR_LIDAR_CKPT=$(cat $W/our_lidar_ckpt_path.txt)
cd $W/mmdetection3d
export PYTHONPATH=$PWD
$PY -W ignore - <<'EOF'
import torch, numpy, mmengine, mmcv, mmdet, mmdet3d, spconv, os
from mmdet3d.utils import register_all_modules
register_all_modules(init_default_scope=True)
from mmdet3d.models.layers.spconv import IS_SPCONV2_AVAILABLE
import projects.BEVFusion.bevfusion
v = dict(numpy=numpy.__version__, torch=torch.__version__, mmengine=mmengine.__version__, mmcv=mmcv.__version__, mmdet=mmdet.__version__, mmdet3d=mmdet3d.__version__, spconv=spconv.__version__)
print(v)
assert v == dict(numpy='1.26.4', torch='2.1.0+cu121', mmengine='0.10.3', mmcv='2.1.0', mmdet='3.3.0', mmdet3d='1.4.0', spconv='2.3.6'), v
assert IS_SPCONV2_AVAILABLE and torch.cuda.is_available()
from mmengine import Config
c = Config.fromfile('projects/BEVFusion/configs/our_bevfusion_mini.py')
import re
pths = re.findall(r"[\w./:\-]+\.pth", c.pretty_text)
print('all .pth strings in config:', pths)
assert c.load_from == os.environ['OUR_LIDAR_CKPT'] and os.path.basename(c.load_from) == 'our_lidar_mini.pth'
assert all(('swin_tiny_patch4_window7_224.pth' in p) or p == c.load_from or p.endswith('our_lidar_mini.pth') for p in pths), pths
assert not any('bevfusion_lidar' in p and 'openmmlab' in p for p in pths)
print('img backbone init:', c.model.img_backbone.init_cfg)
print('train_cfg', c.train_cfg, '| lr', c.optim_wrapper.optimizer.lr, 'accum', c.optim_wrapper.accumulative_counts)
ck = torch.load(c.load_from, map_location='cpu')
print('our LiDAR ckpt tensors:', len(ck['state_dict']), '| note:', ck.get('meta', {}).get('our_note'))
print('STAGE B GATE PASS')
EOF

In [ ]:
%%bash
set -uo pipefail
source /kaggle/working/l3_env.sh
mkdir -p $OUT/logs
trap 'cp -r $W/logs/. $OUT/logs/ 2>/dev/null || true' EXIT
export OUR_LIDAR_CKPT=$(cat $W/our_lidar_ckpt_path.txt)
export OUR_CFG=projects/BEVFusion/configs/our_bevfusion_mini.py OUR_NAME=our_bevfusion_mini OUR_EXPECT_LOAD=our_lidar_mini.pth
export OUR_EMIN=6 OUR_EMAX=24 OUR_BUDGET_S=10800
$PY $W/train_driver.py 2>&1 | tee $OUT/logs/driver_stdout.log
rc=${PIPESTATUS[0]}
echo "driver exit code: $rc"
exit $rc

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/l3_env.sh
D=$OUT/our_bevfusion_mini
OURS=$(find $D/predictions -name results_nusc.json | head -1)
OFF=$(find /kaggle/input -name results_nusc.json -path '*L3D_lidar_camera*' | head -1)
V=$(find /kaggle/input -maxdepth 6 -type d -name v1.0-mini | head -1); ROOT=$(dirname "$V")
echo "ours=$OURS official=$OFF"
mkdir -p $D/demo_images
$PY $W/viz_l3.py --root "$ROOT" --tables "$ROOT/v1.0-mini" --res_lidar "$OURS" --res_cam "$OFF" --out $D/demo_images --every 10 --thr 0.3 --dpi 70 \
  --label_l "OUR LiDAR+Camera (trained on mini TRAIN)" --label_c "official LiDAR+Camera pretrained" \
  --note "left BEV = OUR model, right BEV = official (trained on trainval -> contaminated on mini VAL); mini VAL is NOT a full-nuScenes benchmark" 2>&1 | tail -12
cd $OUT && du -sh . && find . -maxdepth 3 | sort | head -60